## Cosmic Web - JAX - Fusion - SOLUTION

[JAX](https://docs.jax.dev/en/latest/) compiles the same array code with `jax.jit`. It first *traces* the function, recording every operation on *tracers* (placeholders that carry a shape and a dtype). Its compiler, XLA, then *fuses* neighbouring operations into one kernel where it can.

**SOLUTION**

### Table of Contents

1. [Setup](#1-setup)
2. [Stencil](#2-stencil)
3. [Timing](#3-timing)
4. [Fusion](#4-fusion)
5. [Static shapes](#5-static-shapes)
6. [Launch count](#6-launch-count)
7. [Scatter](#7-scatter)
8. [Limitations](#8-limitations)

---

### 1. Setup

By default JAX reserves most of the GPU's memory at its first operation. `XLA_PYTHON_CLIENT_PREALLOCATE=false` turns that off, which leaves room for the profiler.

---

**Quick Docs**

- `jnp.asarray(a)`: returns a JAX array that borrows the CuPy array's memory, so the CuPy array must stay alive.

In [ ]:
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

from functools import partial

import numpy as np
import jax
import jax.numpy as jnp

import cosmic_core as cw

N, K = cw.setup('jax', f'jax {jax.__version__} on {jax.devices()}')
q_dev = cw.by_size(lambda n: jnp.asarray(cw.field(n)))

### 2. Stencil

Let's reuse our previous CuPy code with `cp` replaced by `jnp`. Compile `ecc_hist_jax` below with `K` as a static argument and give each `bincount` its output length.

---

**Quick Docs**

- `partial(jax.jit, static_argnames=('K',))`: compiles the function. `K` becomes a *static* argument: a fixed value compiled into the code. A new value compiles again.
- `jnp.bincount(x, length=K)`: counts each value in `x`. `length` fixes the output size.

In [ ]:
def ecc_cells_jax(q):
    '''Births of the eight cells each voxel owns, as traced array operations.'''
    ex = jnp.minimum(q, jnp.roll(q, -1, 0))
    ey = jnp.minimum(q, jnp.roll(q, -1, 1))
    ez = jnp.minimum(q, jnp.roll(q, -1, 2))
    fxy = jnp.minimum(ex, jnp.roll(ex, -1, 1))
    fxz = jnp.minimum(ex, jnp.roll(ex, -1, 2))
    fyz = jnp.minimum(ey, jnp.roll(ey, -1, 2))
    c = jnp.minimum(fxy, jnp.roll(fxy, -1, 2))
    return q, (ex, ey, ez), (fxy, fxz, fyz), c


@partial(jax.jit, static_argnames=('K',))
def ecc_hist_jax(q, K):
    '''Signed histogram of births (int32, length K), one compiled executable.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_jax(q)
    bc = lambda a: jnp.bincount(a.ravel(), length=K)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)

Why must `length` be a fixed value? Let's try without it:

In [ ]:
try:
    jax.jit(lambda a: jnp.bincount(a.ravel()))(q_dev(N))
except Exception as e:
    print(f'{type(e).__name__}')
    print('  ' + str(e).splitlines()[0][:160])

- Without `length`, the output size is `x.max() + 1`, which depends on the data.
- While tracing, `x` is a tracer that carries only a shape and a dtype.
- XLA must know every shape before it compiles, so the trace stops with an error.

### 3. Timing

JAX dispatches asynchronously: the call returns before the GPU finishes the work.

---

**Quick Docs**

- `jax.block_until_ready(x)`: waits for every array in `x`. Call it before stopping a timer.

In [ ]:
def run_jax(n):
    hist = jax.block_until_ready(ecc_hist_jax(q_dev(n), K=K))
    return np.asarray(hist, dtype=np.int64)

Now let's run a measurement.

In [ ]:
cw.measure('jax_bincount', run_jax, N, prepare=q_dev, note='incl. trace and XLA compile')
cw.compare('jax_bincount', 'cupy')

### 4. Fusion

What did JAX build? We can inspect it at two levels:

- A *jaxpr* (short for JAX expression) is the intermediate representation (IR) of the traced function. It lists the function's *primitives*, JAX's basic operations, in the order the trace recorded them.
- The compiled text (HLO) shows how XLA transformed the IR: each *fusion* compiles a group of primitives into one kernel.

Let's read the jaxpr of one birth and of one count first:

---

**Quick Docs**

- `jax.make_jaxpr(fn, static_argnums=(1,))(q, K)`: returns the jaxpr of `fn`. `fn.__wrapped__` is the Python function under the jit.
- `ecc_hist_jax.lower(q, K=K).compile()`: returns the executable, whose `.as_text()` is the HLO.
- `cw.compile_report(stage, jaxpr, compiled)`: counts the jaxpr's lines, the executable's fusions and scatters, and its temporaries.

In [ ]:
print(jax.make_jaxpr(lambda q: jnp.minimum(q, jnp.roll(q, -1, 0)))(q_dev(N)))   # one birth
print(jax.make_jaxpr(lambda a: jnp.bincount(a.ravel(), length=K))(q_dev(N)))     # one count

The jaxpr reads like the CuPy code: each primitive is a whole-array operation. A *scatter* writes each element to an address given by an index array.

- `roll` runs an inner `jit` of two `slice`s and a `concatenate`.
- The birth takes the `min` of the two arrays.
- `bincount` normalises its indices, then ends in a `scatter-add` that adds one at each birth's bin.
- `unique_indices=False` tells XLA that several births may share a bin.

How many fusions and scatters do you expect for the whole function?

In [ ]:
jaxpr = jax.make_jaxpr(ecc_hist_jax.__wrapped__, static_argnums=(1,))(q_dev(N), K)
compiled = ecc_hist_jax.lower(q_dev(N), K=K).compile()
cw.compile_report('jax_bincount', jaxpr, compiled)

What do we observe?

- The full jaxpr repeats the roll and minimum seven times and the count eight times.
- XLA fuses most roll-and-minimum chains into the scatter that counts them. The three births that the faces and the cube reuse (`ex`, `ey`, `fxy`) get fusions of their own.
- The eight scatters stay separate, one per bincount.
- The temporaries amount to a few full-size arrays, because most intermediates stay inside the fusions.

### 5. Static shapes

The executable is specialised to its input shapes, so each new $N$ compiles again. This means you will repay the JIT compilation cost if the input shapes change.

---

**Quick Docs**

- `jax.clear_caches()`: forgets every compiled executable, so the next call compiles.
- `cw.save_sweep(stage, run, first_call=True, ...)`: times every sweep size, including its first call.

In [ ]:
jax.clear_caches()
cw.save_sweep('jax_bincount', run_jax, first_call=True, prepare=q_dev)
cw.plot_compile_share('jax_bincount', 'first call per size: compile and run',
                      compile_label='first call minus median: trace, compile, first allocations')

- The compile takes about the same time at every size, so it dominates the first call at small sizes.

### 6. Launch count

How do the launches compare with the fusion count of §4, and which kernels take the time?

In [ ]:
trace = cw.trace_call('jax_bincount', run_jax, N, [ecc_cells_jax, ecc_hist_jax], state=[q_dev], header=cw.JAX_HEADER)
scatter_s = lambda rows: sum(d for _, d, name, kind in rows if kind == 'kernel' and 'scatter' in name)
print(f"scatter kernels: {scatter_s(trace['rows']) * 1e3:.2f} ms of {trace['gpu_busy_s'] * 1e3:.2f} ms GPU busy")

- Each fusion launches one kernel, and the eight scatters are among them.
- One copy remains, returning the histogram to the CPU. The scalar reads of `cp.bincount` are gone.
- The scatter kernels take almost all of the GPU's time.

### 7. Scatter

Why do the scatters dominate? A scatter-add performs one atomic add per element into one of $K$ addresses. Adds to the same address contend and are serialised.

Let's split the births into $M$ slabs of $S$ voxels and give each slab its own histogram. The adds then spread over $MK$ addresses. Implement `bc` below.

---

**Quick Docs**

- `a.reshape(-1, S)`: $M = N^3 / S$ slabs of $S$ births.
- `jax.vmap(f)(rows)`: applies `f` to every row as one batched operation.
- `jnp.sum(x, axis=0)`: sums the rows into one histogram.

In [ ]:
@partial(jax.jit, static_argnames=('K', 'S'))
def ecc_hist_jax_slabs(q, K, S=2048):
    '''Signed histogram of births (int32, length K): each slab of S voxels counts into its own row first.'''
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_jax(q)
    bc = lambda a: jnp.sum(jax.vmap(lambda row: jnp.bincount(row, length=K))(a.reshape(-1, S)), axis=0)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)


def run_jax_slabs(n):
    return np.asarray(jax.block_until_ready(ecc_hist_jax_slabs(q_dev(n), K=K)), dtype=np.int64)

Now let's run a measurement.

In [ ]:
cw.measure('jax', run_jax_slabs, N, prepare=q_dev, note='incl. trace and XLA compile')
cw.compare('jax', 'cupy')

trace_slabs = cw.trace_call('jax', run_jax_slabs, N, [ecc_cells_jax, ecc_hist_jax_slabs], state=[q_dev], header=cw.JAX_HEADER)
print(f"scatter kernels: {scatter_s(trace['rows']) * 1e3:.2f} ms before, {scatter_s(trace_slabs['rows']) * 1e3:.2f} ms after")
cw.plot_timelines({'JAX bincount': trace, 'JAX slabs': trace_slabs}, title=f'one call at N = {N}^3: bincount against slabs', highlight='scatter')
cw.save_sweep('jax', run_jax_slabs)

- The scatter-add kernels shrank to a fraction of their time: they now spread over $MK$ addresses instead of $K$.
- The row sums nearly doubled the launch count.
- The scatters still take most of the call.

Takeaways:

- Fusion removed most temporaries and launches, but the scatters' atomic adds still take most of the time.
- XLA fuses the operations that compute a scatter's input into the scatter's kernel. The next operation starts a new kernel.
- `length=` and `S` are static because XLA fixes every shape when it builds the executable.

### 8. Limitations

XLA decides the fusion. We change it by rewriting our function, as the slabs did.

Next: `93__cosmic_web__cccl__iterators.ipynb` describes the stencil to a library algorithm through iterators.